### Checking the Resumes and JD datasets

In [1]:
# Import necessary libraries
import pandas as pd

In [2]:
resume_data_path = "./processed/final_resumes.csv"
resume_data = pd.read_csv(resume_data_path)
resume_data.head()

,ID,Category,resume_md
0,36856210,INFORMATION-TECHNOLOGY,INFORMATION TECHNOLOGY\n\nSummary\n\nDedicated...
1,21780877,INFORMATION-TECHNOLOGY,INFORMATION TECHNOLOGY SPECIALIST GS11\n\nExpe...
2,33241454,INFORMATION-TECHNOLOGY,INFORMATION TECHNOLOGY SUPERVISOR\n\nSummary\n...
3,25990239,INFORMATION-TECHNOLOGY,INFORMATION TECHNOLOGY INSTRUCTOR\n\nSummary\n...
4,16899268,INFORMATION-TECHNOLOGY,INFORMATION TECHNOLOGY MANAGER/ANALYST\n\nProf...


In [3]:
resume_data['Category'].value_counts()

Category
INFORMATION-TECHNOLOGY    120
ENGINEERING               118
DIGITAL-MEDIA              96
Name: count, dtype: int64

In [19]:
# checking first data
resume_data[resume_data['Category']=="DIGITAL-MEDIA"]['resume_md'].iloc[0]

"DIGITAL MEDIA BUYER\n\nProfessional Summary\n\nVersatile digital marketer bringing\n\nHighlights\n\n|  |  |\n| --- | --- |\n| * Pay Per Click (PPC) * Google Adwords * Google Analytics * Content Marketing * Social Media Marketing - Facebook, LinkedIn, Instagram * ROI Reports * MS Office - Excel, Word, Powerpoint, Outlook * PPC Bid Management * Lead Generation * Mobile Marketing * Video Marketing | * SproutSocial * Hootsuite * Marin Software * Drupal * WordPress * HTML * Optimizely * Landing Page Management * A/B Testing * Multivariate Testing * Content Writing * Blogging |\n\nExperience\n\nCompany Name    City  ,   State    Digital Media Buyer   03/2016  to   Current\n\n* Oversees and co-manages PPC campaigns across multiple search engine platforms for three beauty school directory websites.\n* Creates, implements, and manages all organic social profiles and paid social campaigns (Facebook, Instagram, Twitter, Pinterest, etc.) strategies for beauty school directory websites.\n* Organiz

#### Converting MD into Pdf.( As real data will be in pdf)

In [ ]:
# importing libraries
import re
from pathlib import Path
import pandas as pd
from fpdf import FPDF

In [6]:
# Saving few pdfs from md
pdf_output_dir = Path("processed/processed_sample_pdfs")
pdf_output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Class for markdownPDF generator [ Taken from documentation of fpdf and AI reference]
class MarkdownPDFExporter(FPDF):
    """Custom FPDF generator to format Markdown text into clean PDF documents."""

    def __init__(self):
        super().__init__()
        self.set_auto_page_break(auto=True, margin=15)
        self.add_page()
        self.set_font("Helvetica", size=10)

    def add_markdown_content(self, md_text: str):
        for line in str(md_text).split("\n"):
            line = line.strip()
            if not line:
                self.ln(3)
                continue

            # Always reset X position to left margin before processing a line
            self.set_x(self.l_margin)

            # Sanitize UTF-8 symbols for standard Helvetica (Latin-1) font
            clean_line = (
                line.replace("•", "-")
                .replace("—", "-")
                .replace("’", "'")
                .replace("“", '"')
                .replace("”", '"')
            )
            clean_line = clean_line.encode("latin-1", "replace").decode(
                "latin-1"
            )

            # Handle Headers
            if clean_line.startswith("# "):
                self.set_font("Helvetica", style="B", size=14)
                self.multi_cell(
                    0,
                    8,
                    text=clean_line[2:].strip(),
                    new_x="LMARGIN",
                    new_y="NEXT",
                )
                self.set_font("Helvetica", size=10)
            elif clean_line.startswith("## "):
                self.set_font("Helvetica", style="B", size=12)
                self.multi_cell(
                    0,
                    6,
                    text=clean_line[3:].strip(),
                    new_x="LMARGIN",
                    new_y="NEXT",
                )
                self.set_font("Helvetica", size=10)
            elif clean_line.startswith("### "):
                self.set_font("Helvetica", style="B", size=11)
                self.multi_cell(
                    0,
                    5,
                    text=clean_line[4:].strip(),
                    new_x="LMARGIN",
                    new_y="NEXT",
                )
                self.set_font("Helvetica", size=10)

            # Handle Bullet points (Indented)
            elif clean_line.startswith("- ") or clean_line.startswith("* "):
                self.set_x(self.l_margin + 5)  # Indent 5mm from left margin
                bullet_text = re.sub(r"\*\*(.*?)\*\*", r"\1", clean_line[2:])
                avail_w = self.w - self.r_margin - self.x
                self.multi_cell(
                    avail_w,
                    5,
                    text=f"- {bullet_text}",
                    new_x="LMARGIN",
                    new_y="NEXT",
                )

            # Standard Text
            else:
                body_text = re.sub(r"\*\*(.*?)\*\*", r"\1", clean_line)
                self.multi_cell(
                    0, 5, text=body_text, new_x="LMARGIN", new_y="NEXT"
                )


def export_resume_to_pdf(row: pd.Series, output_dir: Path) -> Path:
    """Exports a single dataset row into a PDF file."""
    pdf = MarkdownPDFExporter()
    pdf.add_markdown_content(str(row["resume_md"]))

    file_name = (
        f"resume_{row['ID']}_{row['Category'].lower().replace(' ', '_')}.pdf"
    )
    file_path = output_dir / file_name
    pdf.output(str(file_path))
    return file_path

In [ ]:
# Select a sample resumes across different categories 
sample_df = resume_data.groupby("Category").head(1).reset_index(drop=True)

generated_pdfs = []
print("Generating PDF files from dataset...\n")

for _, row in sample_df.iterrows():
    pdf_file = export_resume_to_pdf(row, pdf_output_dir)
    generated_pdfs.append(pdf_file)
    print(f"Generated: {pdf_file.name}")

print(f"\nPDFs generated: {len(generated_pdfs)}")

Generating PDF files from dataset...
Generated: resume_36856210_information-technology.pdf
Generated: resume_13837784_digital-media.pdf
Generated: resume_14206561_engineering.pdf

Total PDFs generated: 3 in /Users/sujansharma/Documents/0Study_Files/Python-Programming/ResuMatch/data/data_preparation/processed/processed_sample_pdfs
